# IOAI — 2025 Contest Intent Slot Filling (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
os.system('pip -q install -U transformers sentence-transformers')
if not os.path.exists('data/train.conll'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-intent-detection-and-slot-filling', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/train.conll' if os.path.exists('data/train.conll') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Intent & Slot · 모범답안 (XLM-R 결합 미세조정, 영어→러시아어 제로샷)

**다국어 인코더 XLM-RoBERTa** 에 **두 개의 헤드**(문장 [CLS]→intent 분류 / 토큰→BIO 슬롯 분류)를 붙여 **영어
train 으로 결합 학습**하고, 별도 번역 없이 **러시아어 validation 에 제로샷** 적용합니다. XLM-R 의 다국어 표현
덕분에 영어로 배운 슬롯 태깅이 러시아어로 전이됩니다.

검증셋 결과: 베이스라인 0.45 → **모범답안 ≈0.89** (intent_acc ≈0.90, **slot_f1 ≈0.87**). 슬롯 F1 이 0→0.87 로 뛰는 것이 핵심. **GPU 필요**(2 epoch ≈ 6분).

In [ ]:
import os, json, numpy as np, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
def parse_conll(path):
    items, toks, tags, intent = [], [], [], None
    for line in open(path, encoding='utf-8'):
        line = line.rstrip('\n')
        if line.startswith('# intent'): intent = line.split('=' if '=' in line else ':',1)[1].strip()
        elif line.startswith('#'): continue
        elif not line.strip():
            if toks: items.append({'tokens':toks,'intent':intent,'tags':tags}); toks,tags,intent=[],[],None
        else:
            p=line.split('\t')
            if len(p)>=4: toks.append(p[1]); tags.append(p[3])
    if toks: items.append({'tokens':toks,'intent':intent,'tags':tags})
    return items
tr = parse_conll('data/train.conll'); va = parse_conll('data/validation.conll')
intents = sorted(set(x['intent'] for x in tr))
print('train', len(tr), '| val(RU)', len(va), '| intents', len(intents))

In [ ]:
from torch import nn
from transformers import AutoTokenizer, AutoModel
slabels = sorted(set(t for x in tr for t in x['tags'])); s2id={v:i for i,v in enumerate(slabels)}; i2id={v:i for i,v in enumerate(intents)}
tok = AutoTokenizer.from_pretrained('xlm-roberta-base')
class Joint(nn.Module):
    def __init__(self):
        super().__init__(); self.enc = AutoModel.from_pretrained('xlm-roberta-base')
        h = self.enc.config.hidden_size; self.ih = nn.Linear(h, len(intents)); self.sh = nn.Linear(h, len(slabels))
    def forward(self, input_ids, attention_mask):
        o = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.ih(o[:,0]), self.sh(o)
def encode(batch):
    enc = tok([x['tokens'] for x in batch], is_split_into_words=True, truncation=True, max_length=64, padding=True, return_tensors='pt')
    slot = torch.full(enc['input_ids'].shape, -100)
    for bi,x in enumerate(batch):
        prev=None
        for k,w in enumerate(enc.word_ids(bi)):
            if w is None: continue
            if w!=prev: slot[bi,k]=s2id.get(x['tags'][w], s2id['O'])
            prev=w
    return enc, slot


In [ ]:
import random, time
m = Joint().to(dev); opt = torch.optim.AdamW(m.parameters(), lr=2e-5)
ce = nn.CrossEntropyLoss(ignore_index=-100); cei = nn.CrossEntropyLoss()
BS=32; t=time.time()
for ep in range(2):
    m.train(); random.shuffle(tr)
    for i in range(0, len(tr), BS):
        b=tr[i:i+BS]; enc,slot=encode(b); enc={k:v.to(dev) for k,v in enc.items()}; slot=slot.to(dev)
        yi=torch.tensor([i2id[x['intent']] for x in b]).to(dev)
        il,sl=m(enc['input_ids'],enc['attention_mask']); loss=cei(il,yi)+ce(sl.reshape(-1,len(slabels)),slot.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f'epoch {ep} done {round(time.time()-t,1)}s')


In [ ]:
# 러시아어 validation 제로샷 예측 → submission.json
m.eval(); sub=[]; BS=32
with torch.no_grad():
    for i in range(0, len(va), BS):
        b=va[i:i+BS]; enc,_=encode(b); encd={k:v.to(dev) for k,v in enc.items()}
        il,sl=m(encd['input_ids'],encd['attention_mask']); ii=il.argmax(1).cpu().tolist(); ss=sl.argmax(-1).cpu()
        for bi,x in enumerate(b):
            tags=['O']*len(x['tags']); prev=None
            for k,w in enumerate(enc.word_ids(bi)):
                if w is None or w==prev: prev=w; continue
                if w<len(tags): tags[w]=slabels[ss[bi,k].item()]
                prev=w
            sub.append({'intent': intents[ii[bi]], 'tags': tags})
json.dump(sub, open('submission.json','w'))
print('wrote submission.json', len(sub))


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)